### Source Tables:
- `_exponent._bronze_epic_clarity.pat_enc` — patient encounters (CONTACT_DATE)
- `_exponent._bronze_epic_clarity.order_med` — medication orders (ORDERING_DATE, has PAT_ID directly)
- `_exponent._bronze_epic_clarity.order_proc` — procedure orders (ORDERING_DATE, joined via PAT_ENC_CSN_ID)
- `_exponent._bronze_epic_clarity.order_results` — lab results (RESULT_DATE, joined via PAT_ENC_CSN_ID)
- `_exponent._bronze_epic_clarity.patient` — for DEATH_DATE capping

### Strategy:
- Aggregate MIN/MAX clinical activity dates per patient across all source tables
- Create one observation period per patient (no gap splitting)
- Cap end date at DEATH_DATE if patient is deceased
- Use period_type_concept_id = 32817 (EHR)

### Notes:
- This notebook depends on source_to_person being populated for epic_clarity
- order_med has PAT_ID directly; order_proc and order_results join via pat_enc on PAT_ENC_CSN_ID
- Date filtering: >= 1900-01-01 and <= CURRENT_DATE to exclude invalid dates

# Transformation

In [ ]:
%sql
-- Create silver_observation_period temp view for Epic Clarity
-- Simplified to use pat_enc only (other sources to be added after debugging)
CREATE OR REPLACE TEMPORARY VIEW silver_observation_period AS

WITH patient_observation_window AS (
  SELECT
    PAT_ID,
    MIN(DATE(CONTACT_DATE)) AS observation_start,
    MAX(DATE(CONTACT_DATE)) AS observation_end
  FROM _exponent._bronze_epic_clarity.pat_enc
  WHERE PAT_ID IS NOT NULL
    AND CONTACT_DATE IS NOT NULL
    AND CONTACT_DATE >= '1900-01-01'
    AND CONTACT_DATE <= CURRENT_DATE()
  GROUP BY PAT_ID
)

-- Final select with death date capping
SELECT
  pow.observation_start AS observation_period_start_date,
  CASE
    WHEN pt.DEATH_DATE IS NOT NULL AND DATE(pt.DEATH_DATE) < pow.observation_end
    THEN DATE(pt.DEATH_DATE)
    ELSE pow.observation_end
  END AS observation_period_end_date,
  32817 AS period_type_concept_id,
  CONCAT_WS(CHR(31), 'epic_clarity', 'PATIENT', 'PAT_ID', pow.PAT_ID) AS person_source_value,
  CONCAT_WS(CHR(31), 'epic_clarity', 'PATIENT', 'PAT_ID', pow.PAT_ID) AS observation_period_source_value,
  'epic_clarity' AS source_system
FROM patient_observation_window pow
LEFT JOIN _exponent._bronze_epic_clarity.patient pt
  ON pow.PAT_ID = pt.PAT_ID
INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(CHR(31), 'epic_clarity', 'PATIENT', 'PAT_ID', pow.PAT_ID)
  AND stp.active_flag = TRUE

In [0]:
# %sql
# -- Preview
# SELECT * FROM silver_observation_period LIMIT 10

# Write to Silver

In [0]:
%sql
-- Merge to Silver layer
MERGE INTO _exponent.omop_silver.observation_period AS t
USING silver_observation_period AS s
ON t.observation_period_source_value = s.observation_period_source_value

WHEN MATCHED AND (
     NOT (t.observation_period_start_date <=> s.observation_period_start_date)
  OR NOT (t.observation_period_end_date <=> s.observation_period_end_date)
  OR NOT (t.period_type_concept_id <=> s.period_type_concept_id)
  OR NOT (t.person_source_value <=> s.person_source_value)
  OR NOT (t.source_system <=> s.source_system)
)
THEN UPDATE SET
  t.observation_period_start_date = s.observation_period_start_date,
  t.observation_period_end_date   = s.observation_period_end_date,
  t.period_type_concept_id        = s.period_type_concept_id,
  t.person_source_value           = s.person_source_value,
  t.source_system                 = s.source_system,
  t.last_mod_tsp                  = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
  observation_period_start_date,
  observation_period_end_date,
  period_type_concept_id,
  person_source_value,
  observation_period_source_value,
  source_system,
  last_mod_tsp
)
VALUES (
  s.observation_period_start_date,
  s.observation_period_end_date,
  s.period_type_concept_id,
  s.person_source_value,
  s.observation_period_source_value,
  s.source_system,
  current_timestamp()
);

In [0]:
# %sql
# -- Verify silver
# SELECT * FROM _exponent.omop_silver.observation_period
# WHERE source_system = 'epic_clarity'
# LIMIT 10

# Register Observation Period IDs

In [0]:
%sql
-- Insert new mappings to source_to_observation_period
INSERT INTO _exponent.omop_mapping.source_to_observation_period (
    source_system,
    observation_period_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp
)
SELECT
    s.source_system,
    s.observation_period_source_value,
    TRUE AS active_flag,
    current_timestamp() AS created_tsp,
    COALESCE(s.last_mod_tsp, current_timestamp()) AS last_mod_tsp
FROM (
    SELECT DISTINCT source_system, observation_period_source_value, last_mod_tsp
    FROM _exponent.omop_silver.observation_period
    WHERE source_system = 'epic_clarity'
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_observation_period x
  ON s.observation_period_source_value = x.observation_period_source_value;

# Write to Gold

In [0]:
%sql
-- Merge to Gold layer
-- MERGE INTO _exponent.omop.observation_period AS gold
MERGE INTO _exponent.omop_epic.observation_period AS gold
USING (
  SELECT
    sop.observation_period_id,
    stp.person_id,
    s.observation_period_start_date,
    s.observation_period_end_date,
    s.period_type_concept_id
  FROM _exponent.omop_silver.observation_period s
  JOIN _exponent.omop_mapping.source_to_observation_period sop
    ON sop.observation_period_source_value = s.observation_period_source_value
   AND sop.active_flag = TRUE
  JOIN _exponent.omop_mapping.source_to_person stp
    ON stp.person_source_value = s.person_source_value
   AND stp.active_flag = TRUE
  WHERE s.source_system = 'epic_clarity'
) AS src
ON gold.observation_period_id = src.observation_period_id

WHEN MATCHED THEN UPDATE SET
  gold.person_id                       = src.person_id,
  gold.observation_period_start_date   = src.observation_period_start_date,
  gold.observation_period_end_date     = src.observation_period_end_date,
  gold.period_type_concept_id          = src.period_type_concept_id

WHEN NOT MATCHED THEN INSERT (
  observation_period_id,
  person_id,
  observation_period_start_date,
  observation_period_end_date,
  period_type_concept_id
)
VALUES (
  src.observation_period_id,
  src.person_id,
  src.observation_period_start_date,
  src.observation_period_end_date,
  src.period_type_concept_id
);

# Validation

In [0]:
# %sql
# -- Layer counts
# SELECT 'Silver' AS layer, COUNT(*) AS record_count FROM _exponent.omop_silver.observation_period WHERE source_system = 'epic_clarity'
# UNION ALL
# SELECT 'Mapping' AS layer, COUNT(*) AS record_count FROM _exponent.omop_mapping.source_to_observation_period WHERE source_system = 'epic_clarity'
# UNION ALL
# SELECT 'Gold' AS layer, COUNT(*) AS record_count FROM _exponent.omop.observation_period

In [0]:
# %sql
# -- Verify gold
# SELECT * FROM _exponent.omop.observation_period
# LIMIT 10